# Sri Lanka Weather Analytics - ML Dataset Preparation

This notebook prepares the dataset for machine learning prediction of evapotranspiration values in May.

**Requirements: 5.2, 5.3, 5.4, 5.5**

## Steps:
1. Filter data for May months only
2. Select features: precipitation_hours, sunshine_duration, wind_speed_10m_max
3. Define target: et0_fao_evapotranspiration
4. Handle missing values with mean imputation
5. Train/validation split (80/20)
6. Train regression models
7. Evaluate models (RMSE, R²)
8. Predict conditions for May 2026 with ET0 < 1.5mm

In [ ]:
# Import required modules
import os
import sys

# Add spark directory to path for imports
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from spark_data_loader import (
    create_spark_session,
    load_weather_data,
    load_location_data,
    join_weather_location
)

from ml_dataset_preparation import (
    prepare_ml_dataset,
    get_dataset_statistics,
    validate_ml_dataset,
    split_train_validation,
    verify_split_ratios,
    prepare_train_validation_datasets,
    train_linear_regression,
    train_random_forest_regressor,
    evaluate_model,
    print_model_summary,
    predict_conditions_for_low_et0,
    generate_may_2026_prediction,
    print_may_2026_prediction,
    FEATURE_COLUMNS,
    TARGET_COLUMN,
    TRAIN_RATIO,
    VALIDATION_RATIO,
    DEFAULT_SEED,
    TARGET_ET0_THRESHOLD
)

In [ ]:
# Create Spark session
spark = create_spark_session("MLDatasetPreparation")
print(f"Spark version: {spark.version}")

In [ ]:
# Define data paths
weather_path = "../dataset/weatherData.csv"
location_path = "../dataset/locationData.csv"

# Load weather data
print("Loading weather data...")
weather_df = load_weather_data(spark, weather_path)
print(f"Weather records: {weather_df.count()}")

In [ ]:
# Load location data
print("Loading location data...")
location_df = load_location_data(spark, location_path)
print(f"Location records: {location_df.count()}")

In [ ]:
# Join datasets
print("Joining weather and location data...")
joined_df = join_weather_location(weather_df, location_df)
print(f"Joined records: {joined_df.count()}")

## Prepare ML Dataset for May Evapotranspiration Prediction

In [ ]:
# Prepare ML dataset
print("Preparing ML dataset for May evapotranspiration prediction...")
ml_dataset = prepare_ml_dataset(joined_df, use_ml_imputer=False)

print(f"\nFeature columns: {FEATURE_COLUMNS}")
print(f"Target column: {TARGET_COLUMN}")

In [ ]:
# Get dataset statistics
stats = get_dataset_statistics(ml_dataset)

print(f"Total records: {stats['total_records']}")
print(f"\nNull counts after imputation:")
for col, count in stats['null_counts'].items():
    print(f"  {col}: {count}")

In [ ]:
# Validate dataset
validation = validate_ml_dataset(ml_dataset)

print(f"Dataset is valid: {validation['is_valid']}")
print(f"Record count: {validation['record_count']}")

if validation['issues']:
    print("\nIssues found:")
    for issue in validation['issues']:
        print(f"  - {issue}")

In [ ]:
# Show sample data
print("Sample ML dataset:")
ml_dataset.show(10, truncate=False)

In [ ]:
# Show feature statistics
print("Feature Statistics:")
ml_dataset.select(FEATURE_COLUMNS + [TARGET_COLUMN]).describe().show()

## Train/Validation Split (Requirements 5.1)

Split the dataset into 80% training and 20% validation sets using randomSplit with a fixed seed for reproducibility.

In [ ]:
# Perform train/validation split
print(f"Splitting dataset with {TRAIN_RATIO*100:.0f}% training, {VALIDATION_RATIO*100:.0f}% validation")
print(f"Using seed: {DEFAULT_SEED} for reproducibility")

split_result = prepare_train_validation_datasets(ml_dataset, seed=DEFAULT_SEED)

train_df = split_result['train_df']
validation_df = split_result['validation_df']
verification = split_result['verification']
split_stats = split_result['statistics']

In [ ]:
# Display split statistics
print("Split Statistics:")
print(f"  Total records: {split_stats['total_records']}")
print(f"  Training records: {split_stats['training_records']}")
print(f"  Validation records: {split_stats['validation_records']}")
print(f"  Random seed: {split_stats['seed']}")

In [ ]:
# Verify split ratios
print("Split Verification:")
print(f"  Actual train ratio: {verification['actual_train_ratio']:.4f}")
print(f"  Actual validation ratio: {verification['actual_validation_ratio']:.4f}")
print(f"  Expected train ratio: {verification['expected_train_ratio']}")
print(f"  Deviation: {verification['deviation']:.4f}")
print(f"  Within tolerance (±{verification['tolerance']}): {verification['is_valid']}")
print(f"\n{verification['message']}")

## Model Training (Requirements 5.3)

Train Linear Regression and Random Forest Regressor models for evapotranspiration prediction.

In [ ]:
# Train Linear Regression model
print("Training Linear Regression model...")
lr_result = train_linear_regression(
    train_df,
    feature_cols=FEATURE_COLUMNS,
    target_col=TARGET_COLUMN
)

print("\nLinear Regression Model Trained!")
print(f"Coefficients: {lr_result['coefficients']}")
print(f"Intercept: {lr_result['intercept']:.6f}")

In [ ]:
# Train Random Forest Regressor model
print("Training Random Forest Regressor model...")
rf_result = train_random_forest_regressor(
    train_df,
    feature_cols=FEATURE_COLUMNS,
    target_col=TARGET_COLUMN
)

print("\nRandom Forest Model Trained!")
print(f"Feature Importances: {rf_result['feature_importances']}")

## Model Evaluation (Requirements 5.4)

Calculate RMSE and R-squared on validation set for both models.

In [ ]:
# Evaluate Linear Regression on validation set
print("Evaluating Linear Regression on validation set...")
lr_eval = evaluate_model(lr_result, validation_df)

print(f"\nLinear Regression Validation Metrics:")
print(f"  RMSE: {lr_eval['rmse']:.4f}")
print(f"  R²: {lr_eval['r2']:.4f}")
print(f"  MAE: {lr_eval['mae']:.4f}")

In [ ]:
# Evaluate Random Forest on validation set
print("Evaluating Random Forest on validation set...")
rf_eval = evaluate_model(rf_result, validation_df)

print(f"\nRandom Forest Validation Metrics:")
print(f"  RMSE: {rf_eval['rmse']:.4f}")
print(f"  R²: {rf_eval['r2']:.4f}")
print(f"  MAE: {rf_eval['mae']:.4f}")

In [ ]:
# Model Comparison
print("="*60)
print("Model Comparison (Requirements 5.4)")
print("="*60)
print(f"\n{'Model':<25} {'RMSE':<12} {'R²':<12} {'MAE':<12}")
print("-" * 60)
print(f"{'Linear Regression':<25} {lr_eval['rmse']:<12.4f} {lr_eval['r2']:<12.4f} {lr_eval['mae']:<12.4f}")
print(f"{'Random Forest':<25} {rf_eval['rmse']:<12.4f} {rf_eval['r2']:<12.4f} {rf_eval['mae']:<12.4f}")

best_model = "Linear Regression" if lr_eval['rmse'] < rf_eval['rmse'] else "Random Forest"
best_model_result = lr_result if lr_eval['rmse'] < rf_eval['rmse'] else rf_result
best_model_eval = lr_eval if lr_eval['rmse'] < rf_eval['rmse'] else rf_eval
print(f"\nBest model based on RMSE: {best_model}")

In [ ]:
# Show sample predictions
print("Sample Predictions (Linear Regression):")
lr_eval['predictions_df'].select(
    "location_id", "year", 
    *FEATURE_COLUMNS, 
    TARGET_COLUMN, 
    "prediction"
).show(10, truncate=False)

## May 2026 Prediction (Requirements 5.5)

Predict weather conditions that would result in evapotranspiration (ET0) below 1.5mm for May 2026.

In [ ]:
# Predict conditions for May 2026 with ET0 < 1.5mm
print(f"Using {best_model} model for May 2026 prediction...")
print(f"Target: ET0 < {TARGET_ET0_THRESHOLD}mm")

predicted_conditions = predict_conditions_for_low_et0(
    best_model_result,
    train_df,
    target_et0=TARGET_ET0_THRESHOLD
)

print(f"\nPredicted Conditions:")
print(f"  Method: {predicted_conditions['method']}")
print(f"  Precipitation Hours: {predicted_conditions['predicted_precipitation_hours']:.2f}")
print(f"  Sunshine Duration: {predicted_conditions['predicted_sunshine_duration']:.2f}")
print(f"  Wind Speed: {predicted_conditions['predicted_wind_speed']:.2f}")

In [ ]:
# Generate prediction DataFrame for May 2026
may_2026_df = generate_may_2026_prediction(
    spark,
    best_model_result,
    predicted_conditions
)

# Get the predicted ET0 value
predicted_et0 = may_2026_df.select("prediction").collect()[0][0]

print(f"Model Predicted ET0 for May 2026: {predicted_et0:.4f} mm")

In [ ]:
# Print full prediction summary
print_may_2026_prediction(predicted_conditions, predicted_et0)

In [ ]:
# Show the full prediction DataFrame
print("May 2026 Prediction DataFrame:")
may_2026_df.select(
    "year", "month",
    "precipitation_hours", "sunshine_duration", "wind_speed_10m_max",
    "prediction"
).show(truncate=False)

## Summary

The ML pipeline has been completed with the following results:

In [ ]:
# Final Summary
print("="*60)
print("ML Model Training and Prediction Complete!")
print("="*60)
print(f"\nSummary:")
print(f"  - Best Model: {best_model}")
print(f"  - Validation RMSE: {best_model_eval['rmse']:.4f}")
print(f"  - Validation R²: {best_model_eval['r2']:.4f}")
print(f"  - Predicted conditions for May 2026 with ET0 < {TARGET_ET0_THRESHOLD}mm:")
print(f"    * Precipitation Hours: {predicted_conditions['predicted_precipitation_hours']:.2f}")
print(f"    * Sunshine Duration: {predicted_conditions['predicted_sunshine_duration']:.2f}")
print(f"    * Wind Speed: {predicted_conditions['predicted_wind_speed']:.2f}")
print(f"  - Model Predicted ET0: {predicted_et0:.4f} mm")

In [ ]:
# Stop Spark session when done
# spark.stop()